# Imputazione con Mediana per i test ML

In [4]:
import csv
from pathlib import Path
from statistics import median


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        source_dir = candidate / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_ML'
        if source_dir.exists():
            return candidate
    raise FileNotFoundError('Impossibile trovare la radice del progetto con i dati ML.')


project_root = find_project_root()
source_dir = project_root / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_ML'
output_dir = project_root / 'data' / 'processed' / 'Fase3' / 'Imputated_ML'
file_pattern = 'heloc_ML_imputation_test_corrupted_*.csv'


def impute_with_median(input_path: Path, output_path: Path) -> dict:
    with input_path.open(newline='') as csv_file:
        reader = csv.DictReader(csv_file)
        rows = list(reader)
        fieldnames = reader.fieldnames

    if not fieldnames:
        raise ValueError(f'File senza intestazione: {input_path}')

    feature_columns = [column for column in fieldnames if column != 'RiskPerformance']
    medians = {}
    missing_before = 0

    for column in feature_columns:
        values = []
        for row in rows:
            value = row[column]
            if value == '':
                missing_before += 1
            else:
                values.append(float(value))
        medians[column] = median(values)

    for row in rows:
        for column in feature_columns:
            if row[column] == '':
                row[column] = str(medians[column])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', newline='') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return {
        'input_file': input_path.name,
        'output_file': output_path.name,
        'missing_before': missing_before,
        'missing_after': 0,
        'rows': len(rows),
    }


results = []
for input_path in sorted(source_dir.glob(file_pattern)):
    output_name = input_path.name.replace('_corrupted_', '_imputed_')
    output_path = output_dir / output_name
    results.append(impute_with_median(input_path, output_path))

results

[{'input_file': 'heloc_ML_imputation_test_corrupted_MAR_10.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MAR_10.csv',
  'missing_before': 13694,
  'missing_after': 0,
  'rows': 2958},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MAR_25.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MAR_25.csv',
  'missing_before': 29014,
  'missing_after': 0,
  'rows': 2958},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MAR_40.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MAR_40.csv',
  'missing_before': 43926,
  'missing_after': 0,
  'rows': 2958},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MCAR_10.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MCAR_10.csv',
  'missing_before': 13677,
  'missing_after': 0,
  'rows': 2958},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MCAR_25.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MCAR_25.csv',
  'missing_before': 29034,
  'missing_after': 0,
  'rows': 2958},
 {'input_file': 'helo

In [5]:
# Genera un report aggregato dell'imputazione e lo salva in CSV
from pathlib import Path
import csv

# Se le variabili del notebook esistono, riusale; altrimenti ricostruiscile
try:
    _project_root = project_root
    _source_dir = source_dir
    _output_dir = output_dir
except NameError:
    def find_project_root(start: Path | None = None) -> Path:
        current = (start or Path.cwd()).resolve()
        for candidate in [current, *current.parents]:
            source_dir_candidate = candidate / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_ML'
            if source_dir_candidate.exists():
                return candidate
        raise FileNotFoundError('Impossibile trovare la radice del progetto con i dati ML.')
    _project_root = find_project_root()
    _source_dir = _project_root / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_ML'
    _output_dir = _project_root / 'data' / 'processed' / 'Fase3' / 'Imputated_ML'

report_path = _output_dir / 'imputation_report_ML_mediana.csv'
rows_out = []

# Se 'results' è disponibile (variabile creata dalla cella di imputazione), usala direttamente
if 'results' in globals() and isinstance(results, list) and results:
    for r in results:
        rows_out.append({
            'input_file': r.get('input_file'),
            'output_file': r.get('output_file'),
            'rows': r.get('rows'),
            'missing_before': r.get('missing_before'),
            'missing_after': r.get('missing_after')
        })
else:
    # Ricostruisci il report leggendo i file imputati e confrontandoli con gli originali
    for imputed in sorted(_output_dir.glob('heloc_ML_imputation_test_imputed_*.csv')):
        corrupted_name = imputed.name.replace('_imputed_', '_corrupted_')
        corrupted = _source_dir / corrupted_name
        rows_count = 0
        missing_after = 0
        missing_before = 0
        if imputed.exists():
            with imputed.open(newline='') as f:
                reader = csv.DictReader(f)
                imputed_rows = list(reader)
                rows_count = len(imputed_rows)
                # count empty strings after imputation (should be 0)
                for row in imputed_rows:
                    missing_after += sum(1 for v in row.values() if v == '')
        if corrupted.exists():
            with corrupted.open(newline='') as f:
                reader = csv.DictReader(f)
                corrupted_rows = list(reader)
                for row in corrupted_rows:
                    missing_before += sum(1 for v in row.values() if v == '')
        rows_out.append({
            'input_file': corrupted_name if corrupted.exists() else '',
            'output_file': imputed.name,
            'rows': rows_count,
            'missing_before': missing_before,
            'missing_after': missing_after
        })

# Scrivi il report CSV
_report_fields = ['input_file', 'output_file', 'rows', 'missing_before', 'missing_after']
_output_dir.mkdir(parents=True, exist_ok=True)
with report_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=_report_fields)
    writer.writeheader()
    writer.writerows(rows_out)

# Stampa un breve sommario
total_files = len(rows_out)
total_missing_before = sum(r['missing_before'] for r in rows_out)
print(f"Report salvato: {report_path}")
print(f"File processati: {total_files}")
print(f"Valori mancanti prima dell'imputazione (totale): {total_missing_before}")

rows_out  # ritorna il dettaglio come output dell'ultima expression

Report salvato: /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Imputated_ML/imputation_report_ML_mediana.csv
File processati: 9
Valori mancanti prima dell'imputazione (totale): 260571


[{'input_file': 'heloc_ML_imputation_test_corrupted_MAR_10.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MAR_10.csv',
  'rows': 2958,
  'missing_before': 13694,
  'missing_after': 0},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MAR_25.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MAR_25.csv',
  'rows': 2958,
  'missing_before': 29014,
  'missing_after': 0},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MAR_40.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MAR_40.csv',
  'rows': 2958,
  'missing_before': 43926,
  'missing_after': 0},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MCAR_10.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MCAR_10.csv',
  'rows': 2958,
  'missing_before': 13677,
  'missing_after': 0},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MCAR_25.csv',
  'output_file': 'heloc_ML_imputation_test_imputed_MCAR_25.csv',
  'rows': 2958,
  'missing_before': 29034,
  'missing_after': 0},
 {'input_file': 'helo